# 01 - Data Setup: Unity Catalog Infrastructure

This notebook creates the required Unity Catalog resources:
1. **Catalog** and **Schema** for all RAG assets
2. **Volume** for storing source PDF documents
3. **Copy** source documents from workspace to the Volume

> **Run this notebook once** to initialize your environment.
> After setup, upload new documents directly to the Volume for automatic processing.

In [0]:
%run ./00_config

# RAG Chatbot for Project Documentation
## Configuration & Shared Parameters

This notebook defines all shared configuration parameters used across the RAG pipeline.

**Architecture Overview:**
1. **Data Ingestion** (Lakeflow SDP): PDF → Parse → Chunk → Delta Table
2. **Vector Search**: Delta Sync Index with Databricks-managed embeddings
3. **RAG Chain**: Vector Search Retriever + Claude Haiku 4.5 LLM
4. **Serving**: Model Serving Endpoint → Streamlit App on Databricks Apps
5. **Observability**: MLflow Tracing + GenAI Evaluation

✅ Configuration loaded successfully.
   Catalog: ashish_kamboj_rag
   Schema: project_docs_rag
   Volume: /Volumes/ashish_kamboj_rag/project_docs_rag/raw_documents
   Vector Index: ashish_kamboj_rag.project_docs_rag.gold_chunks_index
   Embedding Model: system.ai.qwen3-embedding-0-6b
   LLM Model: system.ai.claude-haiku-4-5


In [0]:
# =============================================================================
# CREATE UNITY CATALOG RESOURCES
# =============================================================================

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")

print(f"✅ Catalog '{CATALOG_NAME}' ready.")
print(f"✅ Schema '{CATALOG_NAME}.{SCHEMA_NAME}' ready.")

✅ Catalog 'ashish_kamboj_rag' ready.
✅ Schema 'ashish_kamboj_rag.project_docs_rag' ready.


In [0]:
# =============================================================================
# CREATE VOLUME FOR RAW DOCUMENTS
# =============================================================================
# The Volume serves as the landing zone for PDF documents.
# File arrival triggers on this Volume will kick off the ingestion pipeline.

spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.{VOLUME_NAME}
    COMMENT 'Raw PDF documents for RAG chatbot ingestion'
""")

print(f"✅ Volume '{VOLUME_PATH}' ready.")

✅ Volume '/Volumes/ashish_kamboj_rag/project_docs_rag/raw_documents' ready.


In [0]:
# =============================================================================
# COPY DOCUMENTS FROM WORKSPACE TO VOLUME
# =============================================================================
# Copy existing PDFs from the workspace folder to the UC Volume.
# After this initial copy, new documents should be uploaded directly to the Volume.

import os
import shutil

# Source: Workspace path (accessible via /Workspace prefix on DBFS)
workspace_source = SOURCE_DOCS_WORKSPACE_PATH

# Destination: Volume path
volume_dest = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}"

# List and copy PDF files
copied_count = 0
for filename in os.listdir(workspace_source):
    if filename.lower().endswith('.pdf'):
        src_path = os.path.join(workspace_source, filename)
        dst_path = os.path.join(volume_dest, filename)
        shutil.copy2(src_path, dst_path)
        copied_count += 1
        print(f"   Copied: {filename}")

print(f"\n✅ Copied {copied_count} PDF files to Volume.")
print(f"   Destination: {volume_dest}")

   Copied: intellecta_genai_rag_blueprint.pdf
   Copied: chronos_timeseries_forecasting_spec.pdf
   Copied: nexacharge_propensity_mlops_spec.pdf
   Copied: sentinelgraph_fraud_detection_blueprint.pdf
   Copied: visioneye_computer_vision_architecture.pdf

✅ Copied 5 PDF files to Volume.
   Destination: /Volumes/ashish_kamboj_rag/project_docs_rag/raw_documents


In [0]:
# =============================================================================
# VERIFY VOLUME CONTENTS
# =============================================================================

files_df = spark.sql(f"""
    SELECT path, size, modification_time 
    FROM list_files('{VOLUME_PATH}')
    WHERE path LIKE '%.pdf'
    ORDER BY path
""")

print(f"📄 Documents in Volume ({files_df.count()} files):")
files_df.display()

📄 Documents in Volume (5 files):


path,size,modification_time
dbfs:/Volumes/ashish_kamboj_rag/project_docs_rag/raw_documents/chronos_timeseries_forecasting_spec.pdf,5233,2026-08-15T11:19:42.000Z
dbfs:/Volumes/ashish_kamboj_rag/project_docs_rag/raw_documents/intellecta_genai_rag_blueprint.pdf,5371,2026-08-15T11:19:41.000Z
dbfs:/Volumes/ashish_kamboj_rag/project_docs_rag/raw_documents/nexacharge_propensity_mlops_spec.pdf,5643,2026-08-15T11:19:42.000Z
dbfs:/Volumes/ashish_kamboj_rag/project_docs_rag/raw_documents/sentinelgraph_fraud_detection_blueprint.pdf,5271,2026-08-15T11:19:42.000Z
dbfs:/Volumes/ashish_kamboj_rag/project_docs_rag/raw_documents/visioneye_computer_vision_architecture.pdf,5553,2026-08-15T11:19:42.000Z
